## Main processing of the data for exploring word embeddings in the context of Ukrainian-English translation

Libraries to install

In [50]:
import importlib
import alignment
importlib.reload(alignment)

<module 'alignment' from 'd:\\UCU\\2_2_yow\\LInAlg\\project\\word-embeddings-la\\scripts\\alignment.py'>

In [51]:
import fasttext
import numpy as np
import pandas as pd

from alignment import (
    learn_least_squares,
    build_candidate_matrices,
    translate_word,
    evaluate
)

from orthogonal import (
    learn_orthogonal
)

from ridge import (
    learn_ridge
)

[FastText](https://fasttext.cc/docs/en/crawl-vectors.html) embedding models for Ukrainian and English

In [6]:
model_uk = fasttext.load_model('../model/cc.uk.300.bin')
model_en = fasttext.load_model('../model/cc.en.300.bin')

Creating pairs of words - Ukrainian and English

In [7]:
def data_to_dictionary(filename: str)-> dict:
    """
    Function creates from the data dictionary for 
    further usage

    :param filename: str, path to the data
    :return: dict, created dictionary
    """
    dict_res = {}
    with open(filename, "r", encoding="utf-8") as f_train:
        next(f_train)
        for line in f_train:
            line = line.strip("\n")
            ua_word, eng_word = line.split(" ")

            if ua_word not in dict_res:
                dict_res.setdefault(ua_word, [])

            dict_res[ua_word].append(eng_word)
    
    return dict_res

In [8]:
train_dict = data_to_dictionary("../data/usage/uk-en-train.csv")
test_dict = data_to_dictionary("../data/usage/uk-en-test.csv")
full_dictionary = data_to_dictionary("../data/usage/uk-en-full.csv")

In [9]:
pairs_train = []

for ua, eng_words in train_dict.items():
    for eng in eng_words:
        pairs_train.append((ua, eng))

Building aligned embeddings matrices for Ukrainian and English words

In [10]:
ua_embeddings, eng_embeddings = [], []

for ua, eng in pairs_train:
    ua_embeddings.append(model_uk.get_word_vector(ua))
    eng_embeddings.append(model_en.get_word_vector(eng))

A_ua, B_eng = np.column_stack(ua_embeddings), np.column_stack(eng_embeddings)

Learning the least Square and Orthogonal Bilingual Mapping

In [11]:
W_ls = learn_least_squares(A_ua, B_eng)
print("Least squares matrix of shape:", W_ls.shape)

Least squares matrix of shape: (300, 300)


Preparing English Candidate words for translation

In [12]:
candidate_words = []
for _, eng_words in full_dictionary.items():
    for eng in eng_words:
        candidate_words.append(eng)

candidate_words, eng_candidate = build_candidate_matrices(candidate_words, model_en)
print("Number of candidate words:", len(candidate_words))
print("Candidate matrix shape:", eng_candidate .shape)

Number of candidate words: 28220
Candidate matrix shape: (28220, 300)


Simple translation of a Ukrainian word

In [23]:
translate_word("кава", W_ls, model_uk, candidate_words, eng_candidate, 5, "cosine")

[('coffee', 0.5656024217605591),
 ('wine', 0.5460401773452759),
 ('chocolate', 0.5318878889083862),
 ('tea', 0.5296329259872437),
 ('drinks', 0.5227132439613342)]

In [ ]:
translate_word("книги", W_ls, model_uk, candidate_words, eng_candidate, 5, "csls")

In [ ]:
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, 5, "euclidean")

### Compare the main realisation with other approaches
In this block, we compare four different approaches for aligning Ukrainian 
and English word embeddings. Each method uses its own transformation matrix W, 
but all methods are evaluated on the same test pairs and the same English candidate words. 

In [47]:
W_least_squares = learn_least_squares(A_ua, B_eng)
W_orthogonal = learn_orthogonal(A_ua, B_eng)
W_ridge = learn_ridge(A_ua, B_eng, lam=0.0001)
W_identity = np.eye(A_ua.shape[0])
print("Least squares matrix of shape:", W_least_squares.shape)
print("Orthogonal matrix shape:", W_orthogonal.shape)
print("Ridge matrix shape:", W_ridge.shape)
print("Identity matrix shape:", W_identity.shape)

Least squares matrix of shape: (300, 300)
Orthogonal matrix shape: (300, 300)
Ridge matrix shape: (300, 300)
Identity matrix shape: (300, 300)


The pairs of ukrainian words and their translation into english

In [18]:
correct_pairs = []

for ua, eng_words in test_dict.items():
    for eng in eng_words:
        correct_pairs.append((ua, eng))
print("Number of test pairs:", len(correct_pairs))
print(correct_pairs[:5])

Number of test pairs: 1827
[('зникли', 'gone'), ('петров', 'petrov'), ('банки', 'banks'), ('банки', 'jars'), ('банки', 'cans')]


Checking the best lambda value for **Ridge Method**

In [54]:
lambda_values = [0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 10]

ridge_lambda_results = []

for lam in lambda_values:
    W_ridge_lam = learn_ridge(A_ua, B_eng, lam=lam)

    top1_accuracy, _ = evaluate(
        test_dict,
        W_ridge_lam,
        model_uk,
        candidate_words,
        eng_candidate,
        top_k=1
    )

    top5_accuracy, _ = evaluate(
        test_dict,
        W_ridge_lam,
        model_uk,
        candidate_words,
        eng_candidate,
        top_k=5
    )

    ridge_lambda_results.append({
        "lambda": lam,
        "top_1_accuracy": top1_accuracy,
        "top_5_accuracy": top5_accuracy
    })

ridge_lambda_df = pd.DataFrame(ridge_lambda_results)
ridge_lambda_df

,lambda,top_1_accuracy,top_5_accuracy
0,0.00000,0.429272,0.605742
1,0.00001,0.429272,0.605742
2,0.00010,0.429272,0.605742
3,0.00100,0.429272,0.605742
4,0.01000,0.429272,0.605742
5,0.10000,0.428571,0.604342
6,1.00000,0.425770,0.601541
7,10.00000,0.374650,0.556723


Evaluate and compare the metchods

In [43]:
print(list(test_dict.items())[:5])

[('зникли', ['gone']), ('петров', ['petrov']), ('банки', ['banks', 'jars', 'cans']), ('олія', ['oil']), ('менші', ['smaller'])]


In [48]:
methods = {
    "No mapping": W_identity,
    "Least squares": W_least_squares,
    "Orthogonal Procrustes": W_orthogonal,
    "Ridge regression": W_ridge,
}
comparison_results = []

for method_name, W in methods.items():
    top1_accuracy, _ = evaluate(
        test_dict,
        W,
        model_uk,
        candidate_words,
        eng_candidate,
        top_k=1
    )
    top5_accuracy, _ = evaluate(
        test_dict,
        W,
        model_uk,
        candidate_words,
        eng_candidate,
        top_k=5
    )
    comparison_results.append({
        "method": method_name,
        "top_1_accuracy": top1_accuracy,
        "top_5_accuracy": top5_accuracy
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,method,top_1_accuracy,top_5_accuracy
0,No mapping,0.000000,0.000000
1,Least squares,0.429272,0.605742
2,Orthogonal Procrustes,0.460784,0.621148
3,Ridge regression,0.429272,0.605742


The **no-mapping** baseline gives 0% accuracy, which confirms that Ukrainian and English FastText embeddings are not directly comparable without alignment. **Least squares** and **ridge regression** both achieve 42.93% Top-1 accuracy and 60.57% Top-5 accuracy. The best method is **Orthogonal Procrustes**, with 46.08% Top-1 accuracy and 62.11% Top-5 accuracy. This suggests that **preserving the geometric structure** of the embedding space is more useful than learning an unrestricted linear mapping for this task.

To analyze the effect of regularization, we evaluated ridge regression for several values of the parameter λ. For small values of λ (from 0 to 0.01), the accuracy remains identical to the least-squares method, with 42.93% Top-1 accuracy and 60.57% Top-5 accuracy. This indicates that in this range the regularization term has almost no influence, and ridge regression behaves the same as least squares.

